In [77]:
import pandas as pd
import numpy as np
import os


### Load the Merged Data
I recreate the merged dataset from Notebook 2.

In [78]:
data_dir = '../data/sql_outputs/'
orders_df = pd.read_csv(os.path.join(data_dir, 'order_features_with_returns.csv'))
customers_df = pd.read_csv(os.path.join(data_dir, 'customer_features.csv'))
products_df = pd.read_csv(os.path.join(data_dir, 'product_features.csv'))

df_merged = pd.merge(orders_df, customers_df, on='customer_unique_id', how='left', suffixes=('', '_cust'))
category_risk_df = products_df.groupby('product_category_name').agg({
    'high_complaint_product': 'mean',
    'high_dissatisfaction_rate': 'mean',
    'avg_review_score': 'mean',
    'low_rating_percentage': 'mean'
}).reset_index().rename(columns={
    'high_complaint_product': 'category_complaint_rate',
    'high_dissatisfaction_rate': 'category_dissatisfaction_rate',
    'avg_review_score': 'category_avg_rating',
    'low_rating_percentage': 'category_low_rating_pct'
})
df_final = pd.merge(df_merged, category_risk_df, left_on='product_category', right_on='product_category_name', how='left')
df_final.drop(columns=['product_category_name'], inplace=True, errors='ignore')
print(f"Initial Shape: {df_final.shape}")


Initial Shape: (96999, 46)


### Temporal Feature Engineering
Raw timestamps (like `2017-10-02 10:56:33`) are useless to an ML algorithm. We must extract the month and day of the week to capture seasonal trends.

In [79]:
df_final['order_purchase_timestamp'] = pd.to_datetime(df_final['order_purchase_timestamp'])
df_final['purchase_month'] = df_final['order_purchase_timestamp'].dt.month
df_final['purchase_day_of_week'] = df_final['order_purchase_timestamp'].dt.dayofweek

print("Extracted Temporal Features:")
print(df_final[['order_purchase_timestamp', 'purchase_month', 'purchase_day_of_week']].head())


Extracted Temporal Features:
  order_purchase_timestamp  purchase_month  purchase_day_of_week
0      2017-10-02 10:56:33              10                     0
1      2018-07-24 20:41:37               7                     1
2      2018-08-08 08:38:49               8                     2
3      2017-11-18 19:28:06              11                     5
4      2018-02-13 21:18:39               2                     1


### Handling Missing Values
ML algorithms crash if they encounter `NaN` values. We will impute numerical categories with the median, and fill edge cases with 0.

In [80]:
cols_to_fill_median = [
    'category_complaint_rate', 
    'category_dissatisfaction_rate', 
    'category_avg_rating', 
    'category_low_rating_pct'
]

for col in cols_to_fill_median:
    df_final[col] = df_final[col].fillna(df_final[col].median())

# Fill any remaining nulls with 0
df_final = df_final.fillna(0)
print(f"Remaining Missing Values: {df_final.isnull().sum().sum()}")


Remaining Missing Values: 0


### Dropping Data Leakage & Identifiers
If we feed `return_probability_score` into the model, the model will cheat and achieve 100% fake accuracy (Data Leakage). We must also drop raw ID strings since they cause overfitting.

In [81]:
df_ml = df_final.copy()

# 1. ENCODE CATEGORICAL COLUMNS (So they are not dropped by models)
# Ordinal mapping for price segments
price_map = {'low': 0, 'medium': 1, 'high': 2}
df_ml['price_segment'] = df_ml['price_segment'].map(price_map).fillna(0)

# Convert high-cardinality strings to numerical categories
categorical_cols = ['product_category', 'customer_state', 'customer_city']
for col in categorical_cols:
    df_ml[col] = df_ml[col].astype('category').cat.codes

# 2. ENHANCE INTERACTIONS
df_ml['delay_freight_cross'] = df_ml['delivery_delay_days'] * df_ml['freight_ratio']
df_ml['review_delay_cross'] = df_ml['review_score'] * df_ml['delivery_delay_days']

# 3. EXPLICITLY DROP IDENTIFIERS, TIMESTAMP STRINGS & REDUNDANT LEAKAGE FLAGS
cols_to_drop = [
    # Unique IDs (Overfitting risk)
    'order_id', 'customer_unique_id', 'product_id', 
    # Raw Timestamps (already represented by days_since_last_order and customer_lifetime_days)
    'order_purchase_timestamp', 'last_order_date', 'first_order_date', 
    # Redundant binary thresholds (Multicollinearity risk)
    'is_unhappy_customer', 'frequent_late_deliveries', 'return_probability_score'
]
df_ml = df_ml.drop(columns=cols_to_drop, errors='ignore')

print(f"Robust dataset shape: {df_ml.shape}")


Robust dataset shape: (96999, 42)


### 6. Save the Final Dataset

In [82]:
os.makedirs('../data/final', exist_ok=True)
df_ml.to_csv('../data/final/ml_ready_dataset.csv', index=False)
print("SUCCESS! Saved ml_ready_dataset.csv to data/final/")


SUCCESS! Saved ml_ready_dataset.csv to data/final/
